# Flood Mapping and Vulnerability Assessment in River and Lagoon Environments

*by Daniele Petrini & Federico Urbini*

## Necessary Imports

In [25]:
import ee
import geemap
import pandas as pd
from pathlib import Path

## Authentication and Folders

In [26]:
ee.Authenticate(auth_mode="localhost")
ee.Initialize(project="floods-2024")
print("Earth Engine initialized correctly.")

Earth Engine initialized correctly.


In [27]:
# Create output folders
Path("outputs").mkdir(exist_ok=True)
Path("outputs/tables").mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/maps").mkdir(parents=True, exist_ok=True)

## Geographic Areas and Dates

In [28]:
# Broad AOI: central-eastern Emilia-Romagna
aoi_emilia_romagna = ee.Geometry.Rectangle([
    10.7, 43.7,   # min longitude, min latitude
    12.8, 45.1    # max longitude, max latitude
])

# Main AOI: Romagna floodplain, September 2024 event
aoi_romagna = ee.Geometry.Rectangle([
    11.3, 44.0,
    12.5, 44.8
])

# Secondary AOI: Bologna / Pianoro, October 2024 event
aoi_bologna = ee.Geometry.Rectangle([
    11.15, 44.15,
    11.65, 44.65
])

In [29]:
# Main places of interest
places = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([11.8833, 44.2856]), {"name": "Faenza"}),
    ee.Feature(ee.Geometry.Point([11.8000, 44.3190]), {"name": "Castel Bolognese"}),
    ee.Feature(ee.Geometry.Point([11.9100, 44.4214]), {"name": "Lugo"}),
    ee.Feature(ee.Geometry.Point([11.9781, 44.4142]), {"name": "Bagnacavallo"}),
    ee.Feature(ee.Geometry.Point([12.2035, 44.4184]), {"name": "Ravenna"}),
    ee.Feature(ee.Geometry.Point([11.3426, 44.4949]), {"name": "Bologna"}),
    ee.Feature(ee.Geometry.Point([11.3436, 44.3896]), {"name": "Pianoro"}),
])

In [30]:
# Event date windows
dates = {
    # September 2024 flood event
    "sept_pre":      ("2024-08-20", "2024-09-15"),
    "sept_post":     ("2024-09-18", "2024-09-25"),
    "sept_recovery": ("2024-09-26", "2024-10-15"),

    # October 2024 flood event
    "oct_pre":       ("2024-10-01", "2024-10-18"),
    "oct_post":      ("2024-10-20", "2024-10-27"),
    "oct_recovery":  ("2024-10-28", "2024-11-15"),

    # Seasonal reference
    "seasonal_ref":  ("2023-08-20", "2023-09-30"),
}

## Utility Functions

In [31]:
def load_sentinel1(aoi, start_date, end_date, orbit_pass=None):
    """
    Load Sentinel-1 GRD SAR data from Earth Engine.

    Filters:
    - AOI
    - date range
    - IW acquisition mode
    - VV and VH polarizations
    - optional orbit pass: 'ASCENDING' or 'DESCENDING'
    """

    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    )

    if orbit_pass is not None:
        collection = collection.filter(
            ee.Filter.eq("orbitProperties_pass", orbit_pass)
        )

    return collection


def load_sentinel2(aoi, start_date, end_date, max_cloud=90):
    """
    Load Sentinel-2 Surface Reflectance Harmonized data from Earth Engine.

    This first notebook only loads and visualizes the dataset.
    Cloud masking and spectral indices will be handled later.
    """

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", max_cloud))
    )

    return collection

## Load Collections and Datasets

In [32]:
# Load Sentinel-1 collections
s1_sept_pre = load_sentinel1(aoi_romagna, *dates["sept_pre"])
s1_sept_post = load_sentinel1(aoi_romagna, *dates["sept_post"])
s1_sept_recovery = load_sentinel1(aoi_romagna, *dates["sept_recovery"])

s1_oct_pre = load_sentinel1(aoi_bologna, *dates["oct_pre"])
s1_oct_post = load_sentinel1(aoi_bologna, *dates["oct_post"])
s1_oct_recovery = load_sentinel1(aoi_bologna, *dates["oct_recovery"])

In [33]:
# Load Sentinel-2 collections
s2_sept_pre = load_sentinel2(aoi_romagna, *dates["sept_pre"], max_cloud=90)
s2_sept_post = load_sentinel2(aoi_romagna, *dates["sept_post"], max_cloud=95)
s2_sept_recovery = load_sentinel2(aoi_romagna, *dates["sept_recovery"], max_cloud=90)

s2_oct_pre = load_sentinel2(aoi_bologna, *dates["oct_pre"], max_cloud=90)
s2_oct_post = load_sentinel2(aoi_bologna, *dates["oct_post"], max_cloud=95)
s2_oct_recovery = load_sentinel2(aoi_bologna, *dates["oct_recovery"], max_cloud=90)

In [34]:
# JRC Global Surface Water
jrc_water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
water_occurrence = jrc_water.select("occurrence")

# ESA WorldCover 2021, 10 m
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")

# Copernicus DEM GLO-30
dem = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM").mosaic()

In [35]:
# Dataset availability check
availability = {
    "S1 September pre-event": s1_sept_pre.size().getInfo(),
    "S1 September post-event": s1_sept_post.size().getInfo(),
    "S1 September recovery": s1_sept_recovery.size().getInfo(),

    "S1 October pre-event": s1_oct_pre.size().getInfo(),
    "S1 October post-event": s1_oct_post.size().getInfo(),
    "S1 October recovery": s1_oct_recovery.size().getInfo(),

    "S2 September pre-event": s2_sept_pre.size().getInfo(),
    "S2 September post-event": s2_sept_post.size().getInfo(),
    "S2 September recovery": s2_sept_recovery.size().getInfo(),

    "S2 October pre-event": s2_oct_pre.size().getInfo(),
    "S2 October post-event": s2_oct_post.size().getInfo(),
    "S2 October recovery": s2_oct_recovery.size().getInfo(),
}

df_availability = pd.DataFrame(
    availability.items(),
    columns=["Dataset window", "Number of images"]
)

df_availability

,Dataset window,Number of images
0,S1 September pre-event,14
1,S1 September post-event,3
2,S1 September recovery,10
3,S1 October pre-event,5
4,S1 October post-event,3
5,S1 October recovery,6
6,S2 September pre-event,50
7,S2 September post-event,9
8,S2 September recovery,20
9,S2 October pre-event,3


In [36]:
# Save availability table
df_availability.to_csv(
    "outputs/tables/01_dataset_availability.csv",
    index=False
)

## Visualization

In [37]:
# Sentinel-1 VV median composites
s1_sept_pre_vv = s1_sept_pre.select("VV").median()
s1_sept_post_vv = s1_sept_post.select("VV").median()

s1_oct_pre_vv = s1_oct_pre.select("VV").median()
s1_oct_post_vv = s1_oct_post.select("VV").median()

# Sentinel-2 median composites
s2_sept_pre_rgb = s2_sept_pre.median()
s2_sept_post_rgb = s2_sept_post.median()

s2_oct_pre_rgb = s2_oct_pre.median()
s2_oct_post_rgb = s2_oct_post.median()

In [38]:
# Visualization parameters
s1_vv_vis = {
    "min": -25,
    "max": 0
}

s2_true_color_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 3000
}

s2_false_color_vis = {
    "bands": ["B8", "B4", "B3"],
    "min": 0,
    "max": 4000
}

water_occurrence_vis = {
    "min": 0,
    "max": 100,
    "palette": ["white", "lightblue", "blue"]
}

dem_vis = {
    "min": 0,
    "max": 1500,
    "palette": ["blue", "green", "yellow", "brown", "white"]
}

worldcover_vis = {
    "min": 10,
    "max": 100,
    "palette": [
        "006400",  # Tree cover
        "ffbb22",  # Shrubland
        "ffff4c",  # Grassland
        "f096ff",  # Cropland
        "fa0000",  # Built-up
        "b4b4b4",  # Bare / sparse vegetation
        "f0f0f0",  # Snow and ice
        "0064c8",  # Permanent water bodies
        "0096a0",  # Herbaceous wetland
        "00cf75",  # Mangroves
        "fae6a0"   # Moss and lichen
    ]
}

## Map 1: Romagna, September

In [39]:
# Map 1: September event, Romagna AOI
Map_sept = geemap.Map(center=[44.35, 11.9], zoom=9)

Map_sept.addLayer(aoi_romagna, {"color": "red"}, "AOI Romagna")
Map_sept.addLayer(places, {"color": "black"}, "Places")

Map_sept.addLayer(
    s1_sept_pre_vv.clip(aoi_romagna),
    s1_vv_vis,
    "S1 VV September pre-event"
)

Map_sept.addLayer(
    s1_sept_post_vv.clip(aoi_romagna),
    s1_vv_vis,
    "S1 VV September post-event"
)

Map_sept.addLayer(
    s2_sept_pre_rgb.clip(aoi_romagna),
    s2_true_color_vis,
    "S2 RGB September pre-event"
)

Map_sept.addLayer(
    s2_sept_post_rgb.clip(aoi_romagna),
    s2_true_color_vis,
    "S2 RGB September post-event"
)

Map_sept.addLayer(
    water_occurrence.clip(aoi_romagna),
    water_occurrence_vis,
    "JRC Water Occurrence"
)

Map_sept.addLayerControl()
Map_sept

Map(center=[44.35, 11.9], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

## Map 2: Bologna, October

In [40]:
# Map 2: October event, Bologna AOI
Map_oct = geemap.Map(center=[44.43, 11.35], zoom=10)

Map_oct.addLayer(aoi_bologna, {"color": "blue"}, "AOI Bologna")
Map_oct.addLayer(places, {"color": "black"}, "Places")

Map_oct.addLayer(
    s1_oct_pre_vv.clip(aoi_bologna),
    s1_vv_vis,
    "S1 VV October pre-event"
)

Map_oct.addLayer(
    s1_oct_post_vv.clip(aoi_bologna),
    s1_vv_vis,
    "S1 VV October post-event"
)

Map_oct.addLayer(
    s2_oct_pre_rgb.clip(aoi_bologna),
    s2_true_color_vis,
    "S2 RGB October pre-event"
)

Map_oct.addLayer(
    s2_oct_post_rgb.clip(aoi_bologna),
    s2_true_color_vis,
    "S2 RGB October post-event"
)

Map_oct.addLayerControl()
Map_oct

Map(center=[44.43, 11.35], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

## Map 3: Auxiliary

In [41]:
# Map 3: Auxiliary datasets
Map_aux = geemap.Map(center=[44.35, 11.8], zoom=8)

Map_aux.addLayer(aoi_emilia_romagna, {"color": "yellow"}, "Broad AOI")
Map_aux.addLayer(aoi_romagna, {"color": "red"}, "AOI Romagna")
Map_aux.addLayer(aoi_bologna, {"color": "blue"}, "AOI Bologna")
Map_aux.addLayer(places, {"color": "black"}, "Places")

Map_aux.addLayer(
    worldcover.clip(aoi_emilia_romagna),
    worldcover_vis,
    "ESA WorldCover"
)

Map_aux.addLayer(
    dem.clip(aoi_emilia_romagna),
    dem_vis,
    "Copernicus DEM"
)

Map_aux.addLayer(
    water_occurrence.clip(aoi_emilia_romagna),
    water_occurrence_vis,
    "JRC Water Occurrence"
)

Map_aux.addLayerControl()
Map_aux

Map(center=[44.35, 11.8], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…